In [2]:
import pandas as pd
df=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\Preprocessed_Data.csv')

In [3]:
df.columns

Index(['Unnamed: 0', 'inningNumber', 'oversActual', 'pitchLine', 'pitchLength',
       'isFour', 'isSix', 'isWicket', 'byes', 'legbyes', 'wides', 'noballs',
       'run', 'totalRuns', 'totalWickets', 'shotType', 'time_of_day',
       'Ground Name', 'Batsman_Name', 'Batsman_Role', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Bowler_Bowling_Style', 'Bowler_Playing_Role'],
      dtype='object')

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import joblib
import warnings
warnings.filterwarnings('ignore')


In [5]:
def load_and_clean(df):
    bool_cols = ['isFour', 'isSix', 'isWicket']
    for c in bool_cols:
        if c in df.columns:
            df[c] = df[c].astype(int)
    df = df.apply(pd.to_numeric, errors='ignore')
    return df

df=load_and_clean(df)
df.head()


,Unnamed: 0,inningNumber,oversActual,pitchLine,pitchLength,isFour,isSix,isWicket,byes,legbyes,...,time_of_day,Ground Name,Batsman_Name,Batsman_Role,Batsman_Batting_Style,Batsman_Playing_Role,Bowler_Name,Bowler_Role,Bowler_Bowling_Style,Bowler_Playing_Role
0,0,1,0.1,1,2,0,0,0,0,0,...,0,1,360,2,1,6,47,1,4,3
1,1,1,0.2,1,4,0,0,0,0,0,...,0,1,360,2,1,6,47,1,4,3
2,2,1,0.3,1,2,0,0,0,0,0,...,0,1,360,2,1,6,47,1,4,3
3,3,1,0.4,1,2,0,0,0,0,0,...,0,1,360,2,1,6,47,1,4,3
4,4,1,0.5,1,2,0,0,1,0,0,...,0,1,360,2,1,6,47,1,4,3


In [6]:
def add_targets(df, goodball_run_threshold=1):
    df = df.copy()
    df['isBoundary'] = ((df.get('isFour', 0) == 1) | (df.get('isSix', 0) == 1)).astype(int)
    df['isGoodBall'] = ((df['run'] <= goodball_run_threshold) & (df['isBoundary'] == 0)).astype(int)
    return df

In [7]:
df = add_targets(df, goodball_run_threshold=1)
print("Counts -> boundaries:", df['isBoundary'].sum(), "good balls:", df['isGoodBall'].sum())

Counts -> boundaries: 3627 good balls: 21285


In [8]:
features = ['inningNumber','oversActual','pitchLine','pitchLength','shotType','time_of_day','Batsman_Batting_Style','Bowler_Bowling_Style']

In [9]:
if 'oversActual' in df.columns:
    df['isDeathOver'] = (df['oversActual'] >= 16).astype(int)
    features.append('isDeathOver')

In [10]:
if 'pitchLine' in df.columns and 'pitchLength' in df.columns:
        df['line_x_length'] = df['pitchLine'] * df['pitchLength']
        features.append('line_x_length')

In [11]:
features

['inningNumber',
 'oversActual',
 'pitchLine',
 'pitchLength',
 'shotType',
 'time_of_day',
 'Batsman_Batting_Style',
 'Bowler_Bowling_Style',
 'isDeathOver',
 'line_x_length']

In [12]:
X = df[features].fillna(0).astype(float)

In [13]:
y=df['isBoundary'].values

In [14]:
X.shape

(27273, 10)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [16]:
classes = np.unique(y_train)
print(len(classes))
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, cw))
print("Class weights:", class_weights)

2
Class weights: {np.int64(0): np.float64(0.5767075491647282), np.int64(1): np.float64(3.759131633356306)}


In [17]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight=class_weights)
rf.fit(X_train_sc, y_train)
y_pred_prob_rf = rf.predict_proba(X_test_sc)[:,1]
y_pred_rf = (y_pred_prob_rf > 0.5).astype(int)
print("RF report:\n", classification_report(y_test, y_pred_rf))
print("Confusion:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_pred_prob_rf))


RF report:
               precision    recall  f1-score   support

           0       0.87      0.96      0.92      4730
           1       0.25      0.08      0.12       725

    accuracy                           0.85      5455
   macro avg       0.56      0.52      0.52      5455
weighted avg       0.79      0.85      0.81      5455

Confusion:
 [[4557  173]
 [ 668   57]]
ROC AUC: 0.6941462418896259


Class imbalance issue:

Only 725 out of 5455 are class 1 → ~13%

RF struggles to predict minority class (boundary / scoring ball)

ROC AUC = 0.694 → okay, shows model can distinguish somewhat but not great.

Precision & recall for class 1:

Precision = 0.25 → only 1 in 4 predicted boundaries is correct

Recall = 0.08 → most actual boundaries are missed

In [18]:
df.columns

Index(['Unnamed: 0', 'inningNumber', 'oversActual', 'pitchLine', 'pitchLength',
       'isFour', 'isSix', 'isWicket', 'byes', 'legbyes', 'wides', 'noballs',
       'run', 'totalRuns', 'totalWickets', 'shotType', 'time_of_day',
       'Ground Name', 'Batsman_Name', 'Batsman_Role', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Bowler_Bowling_Style', 'Bowler_Playing_Role', 'isBoundary',
       'isGoodBall', 'isDeathOver', 'line_x_length'],
      dtype='object')

In [19]:
y=df['isGoodBall']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [21]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [22]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

In [23]:
print("Random Forest Report:")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))

Random Forest Report:
              precision    recall  f1-score   support

           0       0.33      0.16      0.22      1198
           1       0.79      0.91      0.85      4257

    accuracy                           0.74      5455
   macro avg       0.56      0.53      0.53      5455
weighted avg       0.69      0.74      0.71      5455

Confusion Matrix:
 [[ 193 1005]
 [ 399 3858]]
ROC AUC: 0.6624573568899383


The Random Forest model is moderately accurate for detecting good balls, which is sufficient for recommendation purposes, but predicting poor/scoring balls is inherently difficult due to cricket’s stochastic nature and dataset imbalance. The model’s probabilistic outputs are more meaningful than strict classification for real-life use

In [24]:
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [25]:
input_dim = X_train_sc.shape[1]

In [26]:
ann_model = Sequential([
    Dense(64, activation='relu', input_dim=input_dim),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  
])

In [27]:
ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [28]:
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [29]:
ann_model.fit(X_train_sc, y_train,
              validation_split=0.2,
              epochs=50,
              batch_size=64,
              callbacks=[es],
              verbose=1)

Epoch 1/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7535 - loss: 0.5636 - val_accuracy: 0.7793 - val_loss: 0.5305
Epoch 2/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7806 - loss: 0.5370 - val_accuracy: 0.7793 - val_loss: 0.5300
Epoch 3/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5340 - val_accuracy: 0.7793 - val_loss: 0.5300
Epoch 4/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5336 - val_accuracy: 0.7793 - val_loss: 0.5295
Epoch 5/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5317 - val_accuracy: 0.7793 - val_loss: 0.5304
Epoch 6/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7807 - loss: 0.5298 - val_accuracy: 0.7793 - val_loss: 0.5293
Epoch 7/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7807 - loss: 0.5295 - val_accuracy: 0.7793 - val_loss: 0.5290
Epoch 8/50
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7807 - loss: 0.5294 - val_accuracy: 0.

In [30]:
y_prob_ann = ann_model.predict(X_test_sc).flatten()
y_pred_ann = (y_prob_ann > 0.5).astype(int)

171/171 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [31]:
print("ANN Report:")
print(classification_report(y_test, y_pred_ann))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ann))
print("ROC AUC:", roc_auc_score(y_test, y_prob_ann))

ANN Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1198
           1       0.78      1.00      0.88      4257

    accuracy                           0.78      5455
   macro avg       0.39      0.50      0.44      5455
weighted avg       0.61      0.78      0.68      5455

Confusion Matrix:
 [[   0 1198]
 [   0 4257]]
ROC AUC: 0.49925361076698577


Batsman_analysis_engine

In [32]:
df1=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_profiles.csv')

In [33]:
df1.columns

Index(['Full Name', 'Run_Scored', 'Balls_Faced', 'Strike_Rate', 'Dismissals',
       'Dismissal_Rate', 'Boundary_percentage', 'Average'],
      dtype='object')

In [34]:
df.columns

Index(['Unnamed: 0', 'inningNumber', 'oversActual', 'pitchLine', 'pitchLength',
       'isFour', 'isSix', 'isWicket', 'byes', 'legbyes', 'wides', 'noballs',
       'run', 'totalRuns', 'totalWickets', 'shotType', 'time_of_day',
       'Ground Name', 'Batsman_Name', 'Batsman_Role', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Bowler_Bowling_Style', 'Bowler_Playing_Role', 'isBoundary',
       'isGoodBall', 'isDeathOver', 'line_x_length'],
      dtype='object')

In [36]:
df.to_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\Preprocessed_Data.csv')